# Despliegue de Agentes en Producción

Este notebook demuestra los componentes clave para desplegar un agente en producción:
rate limiting, observabilidad y el bucle agéntico con streaming simulado.

In [ ]:
import anthropic
import json
import time
import uuid
import logging
from collections import defaultdict
from functools import wraps
from datetime import datetime

client = anthropic.Anthropic()

# Logging estructurado (JSON)
logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger("agente")

print("Componentes de producción listos")

## 1. Rate Limiter por usuario

In [ ]:
class RateLimiter:
    def __init__(self, max_requests: int = 10, ventana_segundos: int = 60):
        self.max_requests = max_requests
        self.ventana = ventana_segundos
        self._historial: dict = defaultdict(list)

    def permitir(self, user_id: str) -> bool:
        ahora = time.time()
        self._historial[user_id] = [
            t for t in self._historial[user_id]
            if ahora - t < self.ventana
        ]
        if len(self._historial[user_id]) >= self.max_requests:
            return False
        self._historial[user_id].append(ahora)
        return True

    def tiempo_espera(self, user_id: str) -> int:
        historial = self._historial.get(user_id, [])
        if not historial:
            return 0
        return max(0, int(self.ventana - (time.time() - min(historial))))

    def stats(self, user_id: str) -> dict:
        ahora = time.time()
        activos = [t for t in self._historial.get(user_id, []) if ahora - t < self.ventana]
        return {
            "user_id": user_id,
            "requests_en_ventana": len(activos),
            "max_requests": self.max_requests,
            "disponibles": self.max_requests - len(activos),
            "ventana_segundos": self.ventana
        }


# Demo del rate limiter
limiter = RateLimiter(max_requests=5, ventana_segundos=10)

print("Test rate limiting para usuario_a (límite: 5 req/10s):")
for i in range(7):
    permitido = limiter.permitir("usuario_a")
    espera = limiter.tiempo_espera("usuario_a")
    estado = "✓ PERMITIDO" if permitido else f"✗ BLOQUEADO (espera {espera}s)"
    stats = limiter.stats("usuario_a")
    print(f"  Request {i+1}: {estado} | {stats['requests_en_ventana']}/{stats['max_requests']} en ventana")

## 2. Observabilidad: métricas y logging

In [ ]:
class MetricasAgente:
    """Recolector de métricas del agente."""

    def __init__(self):
        self.llamadas = []
        self.herramientas = []

    def registrar_llamada(self, session_id: str, tokens_entrada: int,
                          tokens_salida: int, latencia_ms: float,
                          stop_reason: str):
        self.llamadas.append({
            "ts": datetime.now().isoformat(),
            "session_id": session_id,
            "tokens_entrada": tokens_entrada,
            "tokens_salida": tokens_salida,
            "latencia_ms": latencia_ms,
            "stop_reason": stop_reason
        })

    def registrar_herramienta(self, nombre: str, latencia_ms: float, exito: bool):
        self.herramientas.append({
            "ts": datetime.now().isoformat(),
            "nombre": nombre,
            "latencia_ms": latencia_ms,
            "exito": exito
        })

    def resumen(self) -> dict:
        if not self.llamadas:
            return {"mensaje": "Sin métricas aún"}

        latencias = [c["latencia_ms"] for c in self.llamadas]
        tokens_total = sum(c["tokens_entrada"] + c["tokens_salida"] for c in self.llamadas)
        coste_estimado = tokens_total / 1_000_000 * 0.80  # Haiku ~$0.80/1M tokens promedio

        herr_ok = sum(1 for h in self.herramientas if h["exito"])
        herr_err = len(self.herramientas) - herr_ok

        return {
            "llamadas_api": len(self.llamadas),
            "latencia_media_ms": round(sum(latencias) / len(latencias)),
            "latencia_max_ms": round(max(latencias)),
            "tokens_totales": tokens_total,
            "coste_estimado_usd": round(coste_estimado, 4),
            "herramientas_ok": herr_ok,
            "herramientas_error": herr_err
        }


metricas = MetricasAgente()
print("Sistema de métricas inicializado")

## 3. Agente con observabilidad completa

In [ ]:
HERRAMIENTAS_AGENTE = [
    {"name": "buscar",
     "description": "Busca información en la base de conocimiento",
     "input_schema": {"type": "object",
                      "properties": {"consulta": {"type": "string"}},
                      "required": ["consulta"]}},
    {"name": "calcular",
     "description": "Realiza cálculos matemáticos",
     "input_schema": {"type": "object",
                      "properties": {"expresion": {"type": "string"}},
                      "required": ["expresion"]}}
]


def ejecutar_herramienta_prod(nombre: str, params: dict) -> str:
    """Ejecuta herramienta con métricas."""
    inicio = time.perf_counter()
    exito = True
    try:
        if nombre == "buscar":
            resultado = f"Resultados para '{params['consulta']}': [información relevante encontrada]"
        elif nombre == "calcular":
            resultado = str(eval(params["expresion"], {"__builtins__": {}}))
        else:
            resultado = "Herramienta no disponible"
    except Exception as e:
        resultado = f"Error: {e}"
        exito = False
    finally:
        latencia = (time.perf_counter() - inicio) * 1000
        metricas.registrar_herramienta(nombre, latencia, exito)
        logger.info(json.dumps({
            "evento": "herramienta",
            "nombre": nombre,
            "latencia_ms": round(latencia),
            "exito": exito
        }))
    return resultado


# Almacén de sesiones
SESIONES = {}

def procesar_mensaje(user_id: str, mensaje: str, session_id: str = None) -> dict:
    """Procesa un mensaje con rate limiting, sesiones y métricas."""

    # Rate limiting
    if not limiter.permitir(user_id):
        return {
            "error": f"Rate limit excedido. Espera {limiter.tiempo_espera(user_id)}s",
            "session_id": session_id
        }

    # Gestión de sesión
    session_id = session_id or str(uuid.uuid4())[:8]
    sesion = SESIONES.setdefault(session_id, {"mensajes": [], "tokens_total": 0})
    sesion["mensajes"].append({"role": "user", "content": mensaje})

    herramientas_usadas = []
    tokens_sesion = 0

    inicio_total = time.perf_counter()

    for iteracion in range(10):
        inicio_llamada = time.perf_counter()

        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=800,
            tools=HERRAMIENTAS_AGENTE,
            messages=sesion["mensajes"]
        )

        latencia_llamada = (time.perf_counter() - inicio_llamada) * 1000
        tokens_llamada = resp.usage.input_tokens + resp.usage.output_tokens
        tokens_sesion += tokens_llamada

        metricas.registrar_llamada(
            session_id, resp.usage.input_tokens,
            resp.usage.output_tokens, latencia_llamada, resp.stop_reason
        )

        sesion["mensajes"].append({"role": "assistant", "content": resp.content})

        if resp.stop_reason == "end_turn":
            texto = next((b.text for b in resp.content if hasattr(b, "text")), "")
            break

        if resp.stop_reason == "tool_use":
            resultados = []
            for bloque in resp.content:
                if bloque.type == "tool_use":
                    herramientas_usadas.append(bloque.name)
                    resultado = ejecutar_herramienta_prod(bloque.name, bloque.input)
                    resultados.append({"type": "tool_result",
                                       "tool_use_id": bloque.id,
                                       "content": resultado})
            sesion["mensajes"].append({"role": "user", "content": resultados})

    sesion["tokens_total"] += tokens_sesion
    latencia_total = (time.perf_counter() - inicio_total) * 1000

    logger.info(json.dumps({
        "evento": "chat_completado",
        "session_id": session_id,
        "user_id": user_id,
        "tokens": tokens_sesion,
        "herramientas": herramientas_usadas,
        "latencia_total_ms": round(latencia_total)
    }))

    return {
        "session_id": session_id,
        "respuesta": texto,
        "tokens_usados": tokens_sesion,
        "herramientas_usadas": herramientas_usadas,
        "latencia_ms": round(latencia_total)
    }


# Demo
print("Test del agente con observabilidad:")
print("-" * 50)

r1 = procesar_mensaje("user_123", "¿Cuánto es 15% de 2847?")
print(f"\nSession: {r1['session_id']}")
print(f"Respuesta: {r1['respuesta'][:200]}")
print(f"Tokens: {r1['tokens_usados']} | Latencia: {r1['latencia_ms']}ms")
print(f"Herramientas: {r1['herramientas_usadas']}")

In [ ]:
# Continuar la misma sesión
r2 = procesar_mensaje("user_123", "Y ahora calcula el 21% de IVA sobre ese resultado",
                      session_id=r1['session_id'])
print(f"Respuesta (misma sesión): {r2['respuesta'][:200]}")

# Nueva consulta con búsqueda
r3 = procesar_mensaje("user_456", "Busca información sobre machine learning y explícame los conceptos clave")
print(f"\nNueva sesión: {r3['session_id']}")
print(f"Respuesta: {r3['respuesta'][:200]}")

## 4. Dashboard de métricas

In [ ]:
resumen = metricas.resumen()

print("\n" + "="*50)
print("MÉTRICAS DEL AGENTE")
print("="*50)
for k, v in resumen.items():
    print(f"  {k}: {v}")

print("\n--- Detalle de llamadas ---")
for i, llamada in enumerate(metricas.llamadas):
    print(f"  [{i+1}] session={llamada['session_id'][:8]} | "
          f"{llamada['tokens_entrada']}→{llamada['tokens_salida']} tokens | "
          f"{llamada['latencia_ms']:.0f}ms | {llamada['stop_reason']}")

if metricas.herramientas:
    print("\n--- Herramientas ejecutadas ---")
    for h in metricas.herramientas:
        estado = "✓" if h["exito"] else "✗"
        print(f"  {estado} {h['nombre']} ({h['latencia_ms']:.0f}ms)")

## 5. Checklist de producción

In [ ]:
def verificar_checklist_produccion(agente_config: dict) -> dict:
    """Verifica que el agente cumple con los requisitos de producción."""

    checks = {
        "rate_limiting": agente_config.get("rate_limiter") is not None,
        "limite_iteraciones": agente_config.get("max_iteraciones", 0) > 0,
        "logging_estructurado": agente_config.get("logging") == "json",
        "gestion_sesiones": agente_config.get("sesiones") is not None,
        "metricas": agente_config.get("metricas") is not None,
        "manejo_errores": agente_config.get("error_handler") is not None,
        "timeout_herramientas": agente_config.get("tool_timeout_s", 0) > 0,
    }

    pasados = sum(checks.values())
    total = len(checks)

    return {
        "checks": checks,
        "puntuacion": f"{pasados}/{total}",
        "listo_produccion": pasados == total
    }


config_actual = {
    "rate_limiter": limiter,
    "max_iteraciones": 10,
    "logging": "json",
    "sesiones": SESIONES,
    "metricas": metricas,
    "error_handler": None,       # pendiente
    "tool_timeout_s": 0          # pendiente
}

resultado_checklist = verificar_checklist_produccion(config_actual)

print("CHECKLIST DE PRODUCCIÓN")
print("-" * 40)
for check, ok in resultado_checklist["checks"].items():
    estado = "✓" if ok else "✗"
    print(f"  {estado} {check}")

print(f"\nPuntuación: {resultado_checklist['puntuacion']}")
print(f"Listo para producción: {'Sí' if resultado_checklist['listo_produccion'] else 'No — revisar checks fallidos'}")